In [1]:
# Local Windows setup - no need for drive mounting
import os

# Set the path to your local health documents
folder_path = r"C:\Users\GREEN\Desktop\Health Documents"

# Verify the path exists
if os.path.exists(folder_path):
    print(f"✅ Found directory: {folder_path}")
    pdf_files = [f for f in os.listdir(folder_path) if f.endswith('.pdf')]
    print(f"📄 Found {len(pdf_files)} PDF files")
    for pdf in pdf_files:
        print(f"  - {pdf}")
else:
    print(f"❌ Directory not found: {folder_path}")
    print("Please check the path and make sure the folder exists.")

✅ Found directory: C:\Users\GREEN\Desktop\Health Documents
📄 Found 7 PDF files
  - Bates' Guide to Physical Examination and History Taking.pdf
  - Goodman and Gilmans The Pharmacological Basis of Therapeutic.pdf
  - Harrison's Principles of Internal Medicine.pdf
  - Nelson Essentials of Pediatrics.pdf
  - Oxford Handbook of Clinical Medicine.pdf
  - Robbins and Cotran Pathologic Basis of Disease.pdf
  - WilliamsObstetrics.pdf


In [2]:
# Install required packages for local environment
# Run this cell only if packages are not already installed

import subprocess
import sys

packages = [
    "sentence-transformers",
    "chromadb", 
    "PyMuPDF",
    "tqdm"
]

for package in packages:
    try:
        __import__(package.replace('-', '-'))
        print(f"✅ {package} already installed")
    except ImportError:
        print(f"📦 Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        print(f"✅ {package} installed successfully")

print("🎉 All packages ready!")

📦 Installing sentence-transformers...


KeyboardInterrupt: 

In [3]:
import os
import fitz  # PyMuPDF
from sentence_transformers import SentenceTransformer
import chromadb
from tqdm import tqdm

# Use the local Windows path
folder_path = r"C:\Users\GREEN\Desktop\Health Documents"

print(f"📁 Processing PDFs from: {folder_path}")
print(f"🎯 Target database: medical_knowledge_db (in current directory)")

# List all PDF files in the directory
if os.path.exists(folder_path):
    pdf_files = [f for f in os.listdir(folder_path) if f.endswith('.pdf')]
    print(f"\n📄 Found {len(pdf_files)} PDF files:")
    for i, filename in enumerate(pdf_files, 1):
        full_path = os.path.join(folder_path, filename)
        file_size = os.path.getsize(full_path) / (1024 * 1024)  # MB
        print(f"  {i}. {filename} ({file_size:.1f} MB)")
else:
    print(f"❌ Error: Directory not found!")
    print("Please check the path is correct and the folder exists.")

c:\Users\GREEN\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


📁 Processing PDFs from: C:\Users\GREEN\Desktop\Health Documents
🎯 Target database: medical_knowledge_db (in current directory)

📄 Found 7 PDF files:
  1. Bates' Guide to Physical Examination and History Taking.pdf (5.4 MB)
  2. Goodman and Gilmans The Pharmacological Basis of Therapeutic.pdf (57.3 MB)
  3. Harrison's Principles of Internal Medicine.pdf (13.6 MB)
  4. Nelson Essentials of Pediatrics.pdf (35.6 MB)
  5. Oxford Handbook of Clinical Medicine.pdf (31.5 MB)
  6. Robbins and Cotran Pathologic Basis of Disease.pdf (47.1 MB)
  7. WilliamsObstetrics.pdf (55.3 MB)


In [4]:
import shutil
import os

# Clean up existing database
db_path = "medical_knowledge_db"

if os.path.exists(db_path):
    print(f"🗑️  Removing existing database: {db_path}")
    shutil.rmtree(db_path, ignore_errors=True)
    print("✅ Old database removed")
else:
    print("📁 No existing database found - starting fresh")

# Create the directory structure
os.makedirs(db_path, exist_ok=True)
print(f"📁 Created database directory: {db_path}")

🗑️  Removing existing database: medical_knowledge_db
✅ Old database removed
📁 Created database directory: medical_knowledge_db


In [5]:
def load_pdf_chunks(path, chunk_size=512):
    """
    Extract text chunks from a PDF file.
    
    Args:
        path: Path to the PDF file
        chunk_size: Number of tokens per chunk
    
    Returns:
        List of text chunks
    """
    doc = fitz.open(path)
    chunks = []
    
    print(f"  📖 Processing {doc.page_count} pages...")
    
    for page_num, page in enumerate(doc, 1):
        text = page.get_text()
        if text.strip():  # Only process pages with text
            tokens = text.split()
            for i in range(0, len(tokens), chunk_size):
                chunk = " ".join(tokens[i:i + chunk_size])
                if chunk.strip():  # Only add non-empty chunks
                    chunks.append(chunk.strip())
    
    doc.close()
    print(f"  ✅ Extracted {len(chunks)} chunks")
    return chunks

# Initialize Chroma client for local database
print("🔧 Setting up Chroma database...")
db_path = "medical_knowledge_db"
chroma_client = chromadb.PersistentClient(path=db_path)

# Create or get collection
collection_name = "medical_knowledge"
try:
    # Try to get existing collection
    collection = chroma_client.get_collection(collection_name)
    print(f"📚 Found existing collection: {collection_name}")
    print(f"🔢 Current document count: {collection.count()}")
except:
    # Create new collection if it doesn't exist
    collection = chroma_client.create_collection(collection_name)
    print(f"✨ Created new collection: {collection_name}")

print("✅ Chroma setup complete!")

🔧 Setting up Chroma database...
✨ Created new collection: medical_knowledge
✅ Chroma setup complete!


In [6]:
import glob
import time
from datetime import datetime
from sentence_transformers import SentenceTransformer

# Load the embedding model
print("🤖 Loading all-MiniLM-L6-v2 model...")
start_time = time.time()
model = SentenceTransformer('all-MiniLM-L6-v2')
load_time = time.time() - start_time
print(f"✅ Model loaded in {load_time:.2f} seconds")

# Get PDF files from local directory
folder_path = r"C:\Users\GREEN\Desktop\Health Documents"
book_files = glob.glob(os.path.join(folder_path, "*.pdf"))

print(f"\n🔍 Found {len(book_files)} PDF files to process")

if not book_files:
    print("❌ No PDF files found! Please check:")
    print(f"   - Path exists: {os.path.exists(folder_path)}")
    print(f"   - Path contains PDFs: {folder_path}")
else:
    doc_id_counter = 0  # Ensure unique IDs
    total_chunks = 0
    processing_start = time.time()
    
    for i, book_path in enumerate(book_files, 1):
        book_name = os.path.splitext(os.path.basename(book_path))[0]
        print(f"\n📖 Processing {i}/{len(book_files)}: {book_name}")
        
        try:
            # Extract chunks from PDF
            chunks = load_pdf_chunks(book_path)
            
            if not chunks:
                print(f"  ⚠️  No text extracted from {book_name}")
                continue
                
            print(f"  🧠 Creating embeddings for {len(chunks)} chunks...")
            
            # Create embeddings with progress bar
            embeddings = model.encode(
                chunks, 
                show_progress_bar=True, 
                batch_size=32,
                convert_to_numpy=True
            )
            
            # Prepare metadata
            ids = [f"{book_name}_{j+doc_id_counter}" for j in range(len(chunks))]
            metadatas = [{"source": book_name, "chunk_index": j} for j in range(len(chunks))]
            
            print(f"  💾 Adding to database...")
            
            # Add to collection in batches of 1000 (Chroma limit)
            batch_size = 1000
            for j in range(0, len(chunks), batch_size):
                end_idx = min(j + batch_size, len(chunks))
                
                collection.add(
                    documents=chunks[j:end_idx],
                    metadatas=metadatas[j:end_idx],
                    embeddings=embeddings[j:end_idx].tolist(),
                    ids=ids[j:end_idx]
                )
            
            doc_id_counter += len(chunks)
            total_chunks += len(chunks)
            
            print(f"  ✅ Successfully processed {book_name} ({len(chunks)} chunks)")
            
        except Exception as e:
            print(f"  ❌ Error processing {book_name}: {str(e)}")
            continue
    
    processing_time = time.time() - processing_start
    
    print(f"\n🎉 Processing Complete!")
    print(f"⏱️  Total time: {processing_time:.2f} seconds")
    print(f"📊 Total chunks processed: {total_chunks}")
    print(f"📚 Documents in database: {collection.count()}")
    
    if total_chunks > 0:
        print(f"⚡ Average speed: {total_chunks/processing_time:.1f} chunks/second")
    
    print(f"\n✅ Database saved to: {os.path.abspath('medical_knowledge_db')}")

🤖 Loading all-MiniLM-L6-v2 model...
✅ Model loaded in 8.61 seconds

🔍 Found 7 PDF files to process

📖 Processing 1/7: Bates' Guide to Physical Examination and History Taking
  📖 Processing 430 pages...
  ✅ Extracted 413 chunks
  🧠 Creating embeddings for 413 chunks...


Batches: 100%|██████████| 13/13 [00:44<00:00,  3.43s/it]


  💾 Adding to database...
  ✅ Successfully processed Bates' Guide to Physical Examination and History Taking (413 chunks)

📖 Processing 2/7: Goodman and Gilmans The Pharmacological Basis of Therapeutic
  📖 Processing 1423 pages...
  ✅ Extracted 3164 chunks
  🧠 Creating embeddings for 3164 chunks...


Batches: 100%|██████████| 99/99 [05:02<00:00,  3.06s/it]


  💾 Adding to database...
  ✅ Successfully processed Goodman and Gilmans The Pharmacological Basis of Therapeutic (3164 chunks)

📖 Processing 3/7: Harrison's Principles of Internal Medicine
  📖 Processing 5113 pages...
  ✅ Extracted 6801 chunks
  🧠 Creating embeddings for 6801 chunks...


...

  💾 Adding to database...
  ✅ Successfully processed Oxford Handbook of Clinical Medicine (1276 chunks)

📖 Processing 6/7: Robbins and Cotran Pathologic Basis of Disease
  📖 Processing 924 pages...
  ✅ Extracted 1678 chunks
  🧠 Creating embeddings for 1678 chunks...


Batches: 100%|██████████| 53/53 [02:33<00:00,  2.89s/it]


  💾 Adding to database...
  ✅ Successfully processed Robbins and Cotran Pathologic Basis of Disease (1678 chunks)

📖 Processing 7/7: WilliamsObstetrics
  📖 Processing 1376 pages...
MuPDF error: syntax error: shading colorspace is missing

MuPDF error: syntax error: shading colorspace is missing

MuPDF error: syntax error: shading colorspace is missing

MuPDF error: syntax error: shading colorspace is missing

MuPDF error: syntax error: shading colorspace is missing

MuPDF error: syntax error: shading colorspace is missing

MuPDF error: syntax error: shading colorspace is missing

MuPDF error: syntax error: shading colorspace is missing

MuPDF error: syntax error: shading colorspace is missing

MuPDF error: syntax error: shading colorspace is missing

MuPDF error: syntax error: shading colorspace is missing

MuPDF error: syntax error: shading colorspace is missing

MuPDF error: syntax error: shading colorspace is missing

MuPDF error: syntax error: shading colorspace is missing

MuPDF e

Batches: 100%|██████████| 86/86 [04:17<00:00,  3.00s/it]


  💾 Adding to database...
  ✅ Successfully processed WilliamsObstetrics (2749 chunks)

🎉 Processing Complete!
⏱️  Total time: 1728.60 seconds
📊 Total chunks processed: 17605
📚 Documents in database: 17605
⚡ Average speed: 10.2 chunks/second

✅ Database saved to: c:\Users\GREEN\Projects\Lexrunit\lexcare_ai2\medical_knowledge_db


In [ ]:
# Create a backup zip file of the database
import shutil
import os
from datetime import datetime

db_path = "medical_knowledge_db"
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
zip_name = f"medical_knowledge_db_{timestamp}"

if os.path.exists(db_path):
    print(f"📦 Creating zip archive: {zip_name}.zip")
    
    # Create the zip file
    shutil.make_archive(zip_name, 'zip', db_path)
    
    # Get file size
    zip_path = f"{zip_name}.zip"
    if os.path.exists(zip_path):
        file_size = os.path.getsize(zip_path) / (1024 * 1024)  # Convert to MB
        print(f"✅ Zip created successfully!")
        print(f"📁 File: {zip_path}")
        print(f"📊 Size: {file_size:.2f} MB")
        print(f"💾 Location: {os.path.abspath(zip_path)}")
    else:
        print("❌ Error creating zip file")
else:
    print("❌ Database directory not found!")

In [ ]:
# Test the database by running some sample queries
print("🧪 Testing the created database with sample queries...")

# Load the model (if not already loaded)
if 'model' not in locals():
    print("Loading model...")
    model = SentenceTransformer('all-MiniLM-L6-v2')

# Sample medical queries to test
test_queries = [
    "diabetes symptoms and treatment",
    "heart disease prevention", 
    "high blood pressure medication",
    "chest pain causes",
    "fever and headache"
]

for i, query in enumerate(test_queries, 1):
    print(f"\n🔍 Query {i}: {query}")
    
    # Create query embedding
    query_embedding = model.encode(query)
    
    # Search the database
    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=3,
        include=["documents", "metadatas"]
    )
    
    if results['documents'] and results['documents'][0]:
        print(f"✅ Found {len(results['documents'][0])} results")
        
        # Show first result preview
        first_result = results['documents'][0][0]
        source = results['metadatas'][0][0].get('source', 'Unknown')
        
        preview = first_result[:150].replace('\n', ' ')
        print(f"📄 Source: {source}")
        print(f"📝 Preview: {preview}...")
    else:
        print(f"❌ No results found")

print(f"\n✅ Database test complete!")
print(f"📊 Total documents in database: {collection.count()}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Check database size and provide summary
import os

def get_folder_size(folder_path):
    """Calculate total size of all files in a folder"""
    total_size = 0
    for dirpath, dirnames, filenames in os.walk(folder_path):
        for filename in filenames:
            file_path = os.path.join(dirpath, filename)
            if os.path.exists(file_path):
                total_size += os.path.getsize(file_path)
    return total_size

db_path = "medical_knowledge_db"

if os.path.exists(db_path):
    db_size = get_folder_size(db_path)
    db_size_mb = db_size / (1024 * 1024)
    
    print("📊 DATABASE SUMMARY")
    print("=" * 40)
    print(f"📁 Database path: {os.path.abspath(db_path)}")
    print(f"💾 Database size: {db_size_mb:.2f} MB")
    print(f"📚 Total documents: {collection.count()}")
    
    # List database files
    print(f"\n📋 Database files:")
    for root, dirs, files in os.walk(db_path):
        for file in files:
            file_path = os.path.join(root, file)
            file_size = os.path.getsize(file_path) / (1024 * 1024)
            rel_path = os.path.relpath(file_path, db_path)
            print(f"  📄 {rel_path}: {file_size:.2f} MB")
    
    print(f"\n✅ Ready to use with RAG system!")
    print(f"🔧 Copy the '{db_path}' folder to your project root")
    
else:
    print("❌ Database directory not found!")

Size: 80.62 MB


In [ ]:
# Final steps and instructions
print("🎉 MEDICAL KNOWLEDGE DATABASE CREATION COMPLETE!")
print("=" * 60)

print("\n📋 NEXT STEPS:")
print("1. ✅ Database created successfully in 'medical_knowledge_db' folder")
print("2. 📁 Copy this folder to your LexCare AI project root directory")
print("3. 🔧 Install RAG system dependencies:")
print("   pip install chromadb sentence-transformers")
print("4. 🚀 Run the test script:")
print("   python test_rag_system.py")
print("5. 🌐 Start your server:")
print("   uvicorn main:app --reload")

print("\n💡 TIPS:")
print("• The database uses 'all-MiniLM-L6-v2' embeddings")
print("• Collection name is 'medical_knowledge'")
print("• Chunk size is 512 tokens for optimal balance")
print("• Database is ready for immediate use with the RAG system")

print("\n🔍 To verify integration:")
print("• Check /health/rag endpoint")
print("• Try medical queries through your API")
print("• Monitor cache performance for optimization")

print("\n📞 Need help? Check the RAG_SETUP_GUIDE.md file!")